# SuperPoint, SuperGlue, and LightGlue: PyTorch vs JAX Comparison

## 1. Setup and Imports

In [ ]:
import torch
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import json
import subprocess
import random
from PIL import Image
import sys
from pathlib import Path

# Add the repository root to the path
if 'ipykernel' in sys.modules:
    ROOT_DIR = Path('..')
else:
    ROOT_DIR = Path(__file__).parent.parent.parent

sys.path.append(str(ROOT_DIR))
sys.path.append(str(ROOT_DIR / 'LightGlue'))

from superpoint_jax.model.superpoint_torch import SuperPointTorch
from superpoint_jax.model.superpoint_jax import SuperPointJAX
from superpoint_jax.model.superglue_torch import SuperGlue as SuperGlueTorch
from superpoint_jax.model.superglue_jax import SuperGlueJAX
from superpoint_jax.model.lightglue_jax import LightGlueJAX
from superpoint_jax.utils.convert_to_jax import convert_superpoint_weights, convert_superglue_weights, convert_lightglue_weights
from flax import nnx
from lightglue import LightGlue

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## 2. Load Models and Convert Weights

In [ ]:
# SuperPoint Models
sp_torch = SuperPointTorch(max_num_keypoints=1024).to(device)
weights_path = ROOT_DIR / 'weights/superpoint_torch.pth'
sp_torch.load_state_dict(torch.load(str(weights_path), map_location=device))
sp_torch.eval()

sp_jax = SuperPointJAX(max_num_keypoints=1024, rngs=nnx.Rngs(0))
sp_jax = convert_superpoint_weights(sp_torch, sp_jax)

# SuperGlue Models
sg_torch = SuperGlueTorch({'weights': 'indoor'}).to(device)
sg_torch.eval()

sg_jax = SuperGlueJAX(rngs=nnx.Rngs(0))
sg_jax = convert_superglue_weights(sg_torch, sg_jax)

# LightGlue Model (PyTorch)
lg_torch = LightGlue(features='superpoint').to(device)
lg_torch.eval()

# LightGlue Model (JAX)
lg_jax = LightGlueJAX(rngs=nnx.Rngs(0))
lg_jax = convert_lightglue_weights(lg_torch, lg_jax)

print("All models loaded and converted.")


## 3. Helper Functions for Inference and Visualization

In [ ]:
def load_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load image at {path}")
    img_float = img.astype(np.float32) / 255.0
    return img, img_float

def run_jax_lg(img0_float, img1_float):
    inp0 = jnp.array(img0_float)[None, ..., None]
    inp1 = jnp.array(img1_float)[None, ..., None]

    out0 = sp_jax(inp0, training=False)
    out1 = sp_jax(inp1, training=False)

    v0 = int(out0['valid_counts'][0])
    v1 = int(out1['valid_counts'][0])

    data = {
        'image0': {
            'keypoints': out0['keypoints'][:, :v0],
            'descriptors': out0['descriptors'][:, :v0],
            'image_size': jnp.array([[img0_float.shape[1], img0_float.shape[0]]])
        },
        'image1': {
            'keypoints': out1['keypoints'][:, :v1],
            'descriptors': out1['descriptors'][:, :v1],
            'image_size': jnp.array([[img1_float.shape[1], img1_float.shape[0]]])
        }
    }

    res = lg_jax(data)
    return {
        'kpts0': np.array(data['image0']['keypoints'][0]),
        'kpts1': np.array(data['image1']['keypoints'][0]),
        'matches0': np.array(res['matches0'][0]),
        'matching_scores0': np.array(res['matching_scores0'][0]),
    }

def run_jax_sg(img0_float, img1_float):
    inp0 = jnp.array(img0_float)[None, ..., None]
    inp1 = jnp.array(img1_float)[None, ..., None]

    out0 = sp_jax(inp0, training=False)
    out1 = sp_jax(inp1, training=False)

    v0 = int(out0['valid_counts'][0])
    v1 = int(out1['valid_counts'][0])

    data = {
        'keypoints0': out0['keypoints'][:, :v0],
        'scores0': out0['scores'][:, :v0],
        'descriptors0': out0['descriptors'][:, :v0].transpose(0, 2, 1),
        'image0_shape': (1, 1, *img0_float.shape),
        'keypoints1': out1['keypoints'][:, :v1],
        'scores1': out1['scores'][:, :v1],
        'descriptors1': out1['descriptors'][:, :v1].transpose(0, 2, 1),
        'image1_shape': (1, 1, *img1_float.shape),
    }

    res = sg_jax(data, training=False)
    return {
        'kpts0': np.array(data['keypoints0'][0]),
        'kpts1': np.array(data['keypoints1'][0]),
        'matches0': np.array(res['matches0'][0]),
        'matching_scores0': np.array(res['matching_scores0'][0]),
    }

def run_pytorch_lg(img0_float, img1_float):
    with torch.no_grad():
        inp0 = torch.from_numpy(img0_float).unsqueeze(0).unsqueeze(0).to(device)
        inp1 = torch.from_numpy(img1_float).unsqueeze(0).unsqueeze(0).to(device)

        out0 = sp_torch({'image': inp0})
        out1 = sp_torch({'image': inp1})

        data = {
            'image0': {
                'keypoints': out0['keypoints'][0].unsqueeze(0),
                'descriptors': out0['descriptors'][0].unsqueeze(0),
                'image_size': torch.tensor([[inp0.shape[3], inp0.shape[2]]], device=device).float()
            },
            'image1': {
                'keypoints': out1['keypoints'][0].unsqueeze(0),
                'descriptors': out1['descriptors'][0].unsqueeze(0),
                'image_size': torch.tensor([[inp1.shape[3], inp1.shape[2]]], device=device).float()
            }
        }

        res = lg_torch(data)
        return {
            'kpts0': data['image0']['keypoints'][0].cpu().numpy(),
            'kpts1': data['image1']['keypoints'][0].cpu().numpy(),
            'matches0': res['matches0'][0].cpu().numpy(),
            'matching_scores0': res['matching_scores0'][0].cpu().numpy(),
        }

def visualize_matches(img0, img1, res, title):
    kpts0 = res['kpts0']
    kpts1 = res['kpts1']
    matches0 = res['matches0']
    
    h, w = img0.shape
    composite = np.zeros((h, w * 2 + 10), dtype=np.uint8)
    composite[:, :w] = img0
    composite[:, w+10:] = img1
    composite = cv2.cvtColor(composite, cv2.COLOR_GRAY2BGR)

    valid_indices = np.where(matches0 > -1)[0]
    for idx in valid_indices:
        q_pos = (int(kpts0[idx][0]), int(kpts0[idx][1]))
        t_pos = (int(kpts1[matches0[idx]][0] + w + 10), int(kpts1[matches0[idx]][1]))
        cv2.line(composite, q_pos, t_pos, (0, 255, 0), 1)

    plt.figure(figsize=(15, 7))
    plt.imshow(cv2.cvtColor(composite, cv2.COLOR_BGR2RGB))
    plt.title(f"{title} - {len(valid_indices)} matches")
    plt.axis('off')
    plt.show()


## 4. Run Comparison on Synthetic or Real Data

In [ ]:
# Section 4: Run Comparison on Real Data
dataset_path = ROOT_DIR / 'demo/frames/input_frames/'
frames = sorted([f for f in os.listdir(dataset_path) if f.endswith('.png')])

if len(frames) > 10:
    idx0 = random.randint(0, len(frames) - 11)
    idx1 = idx0 + 10
    img_ref_path = dataset_path / frames[idx0]
    img_target_path = dataset_path / frames[idx1]
    title_suffix = f" (Gap 10: {frames[idx0]} vs {frames[idx1]})"
else:
    # Fallback to synthetic if something went wrong
    img_ref = np.zeros((480, 640), dtype=np.uint8)
    for _ in range(50):
        x, y = random.randint(50, 590), random.randint(50, 430)
        cv2.circle(img_ref, (x, y), 3, 255, -1)
    img_target = cv2.warpAffine(img_ref, cv2.getRotationMatrix2D((320, 240), 2, 1.0), (640, 480))
    img_ref_float = img_ref.astype(np.float32) / 255.0
    img_target_float = img_target.astype(np.float32) / 255.0
    title_suffix = " (Synthetic)"

img_ref, img_ref_float = load_image(img_ref_path)
img_target, img_target_float = load_image(img_target_path)

print(f"Running Comparison on real frames: {frames[idx0]} and {frames[idx1]}")

print("Running LightGlue JAX...")
res_jax_lg = run_jax_lg(img_ref_float, img_target_float)
visualize_matches(img_ref, img_target, res_jax_lg, "LightGlue JAX" + title_suffix)

print("Running SuperGlue JAX...")
res_jax_sg = run_jax_sg(img_ref_float, img_target_float)
visualize_matches(img_ref, img_target, res_jax_sg, "SuperGlue JAX" + title_suffix)

print("Running LightGlue PyTorch...")
res_torch_lg = run_pytorch_lg(img_ref_float, img_target_float)
visualize_matches(img_ref, img_target, res_torch_lg, "LightGlue PyTorch" + title_suffix)


## 5. Comparison Table

In [ ]:

# Auto-generated from benchmark_real_data.py
import time
from pandas import DataFrame
from tabulate import tabulate

results = [{'Implementation': 'LightGlue PyTorch', 'Avg Matches': '183.4', 'Avg Time (ms)': '3041.31', 'Std Time (ms)': '1250.86', 'Min Time (ms)': '2030.22', 'Max Time (ms)': '5274.84'}, {'Implementation': 'LightGlue JAX', 'Avg Matches': '171.0', 'Avg Time (ms)': '9083.23', 'Std Time (ms)': '5383.65', 'Min Time (ms)': '3207.36', 'Max Time (ms)': '17602.91'}, {'Implementation': 'SuperGlue JAX', 'Avg Matches': '119.4', 'Avg Time (ms)': '7836.59', 'Std Time (ms)': '2640.01', 'Min Time (ms)': '4903.92', 'Max Time (ms)': '11279.98'}]
df = DataFrame(results)
print("\nComparison Table (Averaged over random pairs):")
print(tabulate(df, headers='keys', tablefmt='pipe', showindex=False))
